# Jonas Thamane 第 1 周练习 —— PR / 技术问答解释器

## 练习目标（理念）

为展示你对 **OpenAI API** 与本地 **Ollama**（经 OpenAI 兼容接口）的熟悉程度，请构建一个小工具：

- **输入**：一个技术问题（例如「这段 Python 代码在干什么？」）
- **输出**：清晰、分步、偏教学向的解释
- **额外要求**：用**流式（streaming）**一边生成一边更新 Markdown 显示

这是你在课程期间自己也能天天用的「技术导师」工具。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `client.chat.completions.create(...)` |
| `messages`（system / user） | `build_messages()` 组装 system + user |
| 流式输出 `stream=True` | 边收 `delta.content` 边 `update_display` |
| 切换后端 | `PROVIDER` / `get_client()`：`openai` 或 `ollama` |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `OPENAI_API_KEY`（OpenAI 默认客户端会读它）
3. 若要用本地模型：先启动 Ollama，并 `ollama pull llama3.2`；把 `PROVIDER` 改成 `"ollama"`
4. 在最后一格改写 `question`，再调用 `ask_tech_tutor`


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：Markdown 渲染、display、流式时用 update_display 原地刷新
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI 客户端类：既可调云端，也可指向本地 Ollama 的 OpenAI 兼容端点
from openai import OpenAI
# 导入标准库 os：读环境变量（Environment Variables）；本格虽未直接用，但常与 dotenv 配套
import os


In [ ]:
# ========== 环境与模型常量：密钥加载 + 模型名集中管理 ==========

# 加载 .env；override=True 表示用 .env 覆盖进程里已有的同名环境变量
load_dotenv(override=True)

# OpenAI 云端小模型：便宜、够用，适合解释类问答（字符串必须是真实 model id）
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2；须与本机已安装名称一致
MODEL_LLAMA = 'llama3.2'

# 默认后端提供方：改成 "ollama" 就会走本地；保持英文字符串，因 get_client 用它做分支判断
PROVIDER = "openai"


In [ ]:
# ========== 客户端工厂：按 provider 返回可用的 OpenAI 风格客户端 ==========

def get_client(provider: str):
    """根据 provider 名字返回 OpenAI 客户端：云端官方，或指向本地 Ollama /v1。"""
    # 云端：无参 OpenAI() 会默认从环境变量 OPENAI_API_KEY 读取密钥
    if provider == "openai":
        return OpenAI()

    # 本地 Ollama：把 base_url 指到 OpenAI 兼容接口；api_key 可任意非空（Ollama 通常不校验）
    elif provider == "ollama":
        return OpenAI(
            base_url="http://localhost:11434/v1",
            api_key="ollama"
        )

    # 未知提供方：直接抛错，避免静默用错后端
    else:
        raise ValueError("Unsupported provider")


In [ ]:
# ========== 组装 messages：system 定角色，user 放具体问题 ==========

def build_messages(question: str):
    """把技术问题包装成 Chat Completions 需要的 messages 列表。"""

    # system prompt 保留英文：这是发给模型的指令，改译会改变回答风格/行为
    system_prompt = """
    You are an expert software engineer and technical educator.
    Your task is to:
    
    1. Explain the code clearly
    2. Break down each component
    3. Explain design reasoning
    4. Mention edge cases
    5. Suggest improvements
    6. Provide a short example if useful
    """

    # user prompt：把具体 question 嵌进模板；发给模型的英文指令保留原样
    user_prompt = f"""
    Please explain the following technical question in depth:
    
    {question}
    """

    # 返回标准两段式 messages：先 system，后 user
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]


In [ ]:
# ========== 核心：流式提问并在笔记本里实时刷新 Markdown ==========

def ask_tech_tutor(question: str, provider="openai"):
    """向选定后端发流式请求，边生成边 update_display，最后返回完整回答字符串。"""

    # 按 provider 拿到客户端（OpenAI 或指向 Ollama 的兼容客户端）
    client = get_client(provider)
    # 选模型：openai → MODEL_GPT；否则用本地 MODEL_LLAMA
    model = MODEL_GPT if provider == "openai" else MODEL_LLAMA
    # 组装 system + user messages
    messages = build_messages(question)

    # stream=True：服务端持续推送增量，而不是等整段生成完
    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )

    # response：把各块 delta 拼成完整答案
    response = ""
    # 先放一个空的 Markdown 显示位，记下 display_id，后面用同一 id 原地更新
    display_handle = display(Markdown(""), display_id=True)

    # 遍历流式事件；每个 chunk 可能带一小段文本
    for chunk in stream:
        # 增量文本通常在 choices[0].delta.content；可能为 None（角色/结束事件）
        delta = chunk.choices[0].delta.content
        if delta:
            response += delta
            # 用同一 display_id 刷新，实现「打字机」效果
            update_display(Markdown(response), display_id=display_handle.display_id)

    # 返回完整字符串，便于后续再处理或打印
    return response


In [ ]:
# ========== 提问：改这里的 question / PROVIDER 就能换问题或后端 ==========

# 发给模型的技术问题保持英文（可运行 / 影响回答的字符串不翻译）
question = """
Explain what this Python code does and why:

yield from {book.get("author") for book in books if book.get("author")}
"""

# 调用技术导师；provider 用上面的 PROVIDER 常量（"openai" 或 "ollama"）
ask_tech_tutor(question, provider=PROVIDER)
